# LeetCode 957 — Prison Cells After N Days bằng Python

Link đề: https://leetcode.com/problems/prison-cells-after-n-days/ · Độ khó: **Medium** · Chủ đề: Array (mảng), Hash Table (bảng băm), Math (toán), Bit Manipulation (thao tác bit)

Skeleton LeetCode:

```python
class Solution:
    def prisonAfterNDays(self, cells: List[int], n: int) -> List[int]:
        ...
```

Ghi chú: đây là bài **mô phỏng với `n` rất lớn** (`10^9`). Trạng thái chỉ là **8 bit** nên có tối đa `256` trạng thái; trạng thái ở ngày kế tiếp chỉ phụ thuộc trạng thái hiện tại, vì vậy dãy trạng thái **lặp lại có chu kỳ** nhanh chóng. Ý tưởng chính: **phát hiện chu kỳ** (bằng bảng băm trạng thái -> ngày đầu tiên) rồi nhảy qua phần lặp. Thực tế còn có nhận xét mạnh hơn: từ ngày `1` trở đi hai ô đầu/cuối luôn trống nên chỉ còn `6` bit, và chu kỳ chia hết cho `14`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: 8 ô tù; mỗi ngày ô `i` (với `1 <= i <= 6`) có người nếu hai ô kề **giống nhau**, ngược lại trống; ô đầu và ô cuối **không có hai ô kề** nên luôn trống từ ngày `1`.
2. Nắm các công cụ Python cần thiết: `tuple` làm khóa từ điển, mã hóa trạng thái thành số nguyên 8 bit, toán tử bit `<<`, `>>`, `^`, `~`, `&`, và phép chia lấy dư để nhảy qua chu kỳ.
3. Hiểu nguyên lý: hàm chuyển trạng thái `f` trên tập hữu hạn ⇒ dãy `s, f(s), f(f(s)), ...` **cuối cùng tuần hoàn**; biết ngày vào chu kỳ `a` và chu kỳ `p` thì `f^n(s) = f^(a + (n - a) mod p)(s)` với `n >= a`.
4. Đi qua 4 cách: mô phỏng từng ngày (brute, chỉ chạy được `n` nhỏ), phát hiện chu kỳ bằng bảng băm (cách nộp chính), nhảy `n -> (n - 1) mod 14 + 1` nhờ nhận xét chu kỳ, và mô phỏng bằng thao tác bit kết hợp phát hiện chu kỳ; đối chiếu với oracle **lũy thừa hàm bằng bình phương liên tiếp** và kiểm tra vét cạn mọi `256` trạng thái.
5. Tránh các lỗi kinh điển: không đặt ô đầu/cuối về `0`, cập nhật tại chỗ (dùng giá trị đã đổi), dùng `n % 14` thô, mô phỏng `10^9` ngày, và lưu tham chiếu danh sách thay vì bản sao khi phát hiện chu kỳ.

## 2. Đề bài & ví dụ

Có `8` ô tù xếp thành hàng, mỗi ô có người (`1`) hoặc trống (`0`). Mỗi ngày trạng thái đổi theo quy tắc:

- nếu một ô có **hai ô kề đều có người hoặc đều trống** thì ô đó **có người**;
- ngược lại nó **trống**.

Lưu ý: vì là một hàng nên ô đầu và ô cuối **không có hai ô kề**.

Cho mảng `cells` (`cells[i] == 1` nếu ô `i` có người) và số nguyên `n`, trả về trạng thái nhà tù sau `n` ngày.

**Ví dụ 1:** `cells = [0,1,0,1,1,0,0,1]`, `n = 7` -> `[0,0,1,1,0,0,0,0]` (bảng từng ngày có trong đề; ngày 1: `[0,1,1,0,0,0,0,0]`).

**Ví dụ 2:** `cells = [1,0,0,1,0,0,1,0]`, `n = 1000000000` -> `[0,0,1,1,1,1,1,0]`.

**Ràng buộc:**

- `cells.length == 8`
- `cells[i]` là `0` hoặc `1`.
- `1 <= n <= 10^9`

In [1]:
from typing import List
import random, time

examples = [
    ([0, 1, 0, 1, 1, 0, 0, 1], 7, [0, 0, 1, 1, 0, 0, 0, 0]),
    ([1, 0, 0, 1, 0, 0, 1, 0], 10 ** 9, [0, 0, 1, 1, 1, 1, 1, 0]),
]
for cells, n, expected in examples:
    print(f"cells={cells} n={n} kỳ vọng={expected}")

cells=[0, 1, 0, 1, 1, 0, 0, 1] n=7 kỳ vọng=[0, 0, 1, 1, 0, 0, 0, 0]
cells=[1, 0, 0, 1, 0, 0, 1, 0] n=1000000000 kỳ vọng=[0, 0, 1, 1, 1, 1, 1, 0]


## 3. Kiến thức nền

### 3.1. Một ngày trôi qua

Ô `i` (`1 <= i <= 6`) trở thành `1` nếu `cells[i - 1] == cells[i + 1]`, ngược lại `0`. Ô `0` và ô `7` luôn thành `0` (không có đủ hai ô kề). Quan trọng: tính mọi ô mới từ trạng thái **cũ** (không cập nhật tại chỗ).

In [2]:
def next_day(cells: List[int]) -> List[int]:
    new = [0] * 8
    for i in range(1, 7):
        new[i] = 1 if cells[i - 1] == cells[i + 1] else 0
    return new

state = [0, 1, 0, 1, 1, 0, 0, 1]
for day in range(1, 8):
    state = next_day(state)
    print(f"ngày {day}:", state)
assert state == [0, 0, 1, 1, 0, 0, 0, 0]

ngày 1: [0, 1, 1, 0, 0, 0, 0, 0]
ngày 2: [0, 0, 0, 0, 1, 1, 1, 0]
ngày 3: [0, 1, 1, 0, 0, 1, 0, 0]
ngày 4: [0, 0, 0, 0, 0, 1, 0, 0]
ngày 5: [0, 1, 1, 1, 0, 1, 0, 0]
ngày 6: [0, 0, 1, 0, 1, 1, 0, 0]
ngày 7: [0, 0, 1, 1, 0, 0, 0, 0]


### 3.2. Phát hiện chu kỳ

Hàm `f` biến một trạng thái thành trạng thái ngày kế tiếp; có hữu hạn trạng thái nên dãy `s0, f(s0), f(f(s0)), ...` sớm gặp lại một trạng thái đã thấy. Nếu trạng thái xuất hiện lần đầu ở ngày `a` và lặp lại ở ngày `b` thì chu kỳ `p = b - a`; mọi ngày `d >= a` có trạng thái bằng trạng thái ngày `a + (d - a) mod p`.

### 3.3. Mã hóa trạng thái bằng 8 bit

Bit `i` của số nguyên `s` là ô `i`. Khi đó `s << 1` đặt ô `i - 1` vào bit `i` và `s >> 1` đặt ô `i + 1` vào bit `i`. Hai ô kề của ô `i` giống nhau ⇔ `bit_i(s << 1) == bit_i(s >> 1)` ⇔ `bit_i((s << 1) ^ (s >> 1)) == 0`. Vậy `next = ~((s << 1) ^ (s >> 1)) & 0b01111110`: giữ lại bit `1..6`, ép bit `0` và `7` bằng `0`.

In [3]:
def to_int(cells: List[int]) -> int:
    return sum(c << i for i, c in enumerate(cells))

def to_cells(s: int) -> List[int]:
    return [(s >> i) & 1 for i in range(8)]

def step_bits(s: int) -> int:
    return ~((s << 1) ^ (s >> 1)) & 0b01111110

cells = [0, 1, 0, 1, 1, 0, 0, 1]
s = to_int(cells)
print(f"{s:08b}", to_cells(s))
assert to_cells(step_bits(s)) == next_day(cells) == [0, 1, 1, 0, 0, 0, 0, 0]
# kiểm tra mọi 256 trạng thái
assert all(to_cells(step_bits(x)) == next_day(to_cells(x)) for x in range(256))
print("bản bit và bản danh sách khớp trên mọi 256 trạng thái")

10011010 [0, 1, 0, 1, 1, 0, 0, 1]
bản bit và bản danh sách khớp trên mọi 256 trạng thái


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 1 sang ngày 1: `[0,1,0,1,1,0,0,1]`:

- ô `1`: hai ô kề là `cells[0] = 0` và `cells[2] = 0`, giống nhau ⇒ `1`;
- ô `2`: `cells[1] = 1` và `cells[3] = 1` ⇒ `1`;
- ô `3`: `cells[2] = 0` và `cells[4] = 1` khác nhau ⇒ `0`;
- ô `4`: `cells[3] = 1`, `cells[5] = 0` ⇒ `0`;
- ô `5`: `cells[4] = 1`, `cells[6] = 0` ⇒ `0`;
- ô `6`: `cells[5] = 0`, `cells[7] = 1` ⇒ `0`;
- ô `0` và ô `7`: `0`.

Kết quả `[0,1,1,0,0,0,0,0]`, đúng bảng đề.

Với `n = 10^9`, không thể mô phỏng từng ngày; nhưng chỉ có `256` trạng thái nên dãy lặp lại trong tối đa `256` ngày: ta chỉ mô phỏng tới khi gặp trạng thái cũ rồi suy ra ngày `n` bằng phép lấy dư.

## 5. Cách 1 — Brute Force: mô phỏng từng ngày

**Ý tưởng.** Lặp `n` lần `cells = next_day(cells)`. Đúng nhưng chỉ khả thi khi `n` nhỏ; là **oracle** cho `n` nhỏ.

In [4]:
def prison_brute(cells: List[int], n: int) -> List[int]:
    cells = list(cells)
    for _ in range(n):
        cells = next_day(cells)
    return cells

got = prison_brute(*examples[0][:2])
print(got, "OK" if got == examples[0][2] else "SAI")
assert got == examples[0][2]

[0, 0, 1, 1, 0, 0, 0, 0] OK


**Độ phức tạp và nút thắt.** `O(8 * n)`: với `n = 10^9` là `8 * 10^9` thao tác. Xem thời gian tăng tuyến tính theo `n`:

In [5]:
for n in (10 ** 3, 10 ** 4, 10 ** 5, 10 ** 6):
    t0 = time.perf_counter()
    prison_brute(examples[1][0], n)
    dt = time.perf_counter() - t0
    print(f"n={n:8}: {1000 * dt:9.1f} ms  -> ước lượng cho n = 10^9: {dt * 10 ** 9 / n / 3600:7.1f} giờ")

n=    1000:       0.7 ms  -> ước lượng cho n = 10^9:     0.2 giờ
n=   10000:       6.8 ms  -> ước lượng cho n = 10^9:     0.2 giờ
n=  100000:      46.0 ms  -> ước lượng cho n = 10^9:     0.1 giờ


n= 1000000:     515.8 ms  -> ước lượng cho n = 10^9:     0.1 giờ


Với `n = 10^9` phải mất hàng chục giờ. Nút thắt: ta mô phỏng cả những ngày **lặp lại** trạng thái đã gặp.

## 6. Ý tưởng thuật toán / bước đột phá

**Trạng thái chỉ có 256 giá trị.** Ghi lại ngày đầu tiên mỗi trạng thái xuất hiện; khi gặp lại một trạng thái ở ngày `b` mà lần đầu ở ngày `a`, ta có chu kỳ `p = b - a` và có thể bỏ qua `(n - b) // p` chu kỳ đầy đủ, rồi mô phỏng nốt phần dư.

Nhận xét mạnh hơn (kiểm chứng ở mục 10 bằng cách duyệt mọi `256` trạng thái): sau ngày `1` hai bit đầu/cuối luôn `0`, chỉ còn `6` bit nên `<= 64` trạng thái, và chu kỳ **chia hết cho `14`**. Do đó `f^n(s) = f^(((n - 1) mod 14) + 1)(s)` với mọi `n >= 1`.

| Cách | Ý tưởng |
|---|---|
| 2 | Bảng băm trạng thái -> ngày, nhảy qua chu kỳ (nộp) |
| 3 | `n = (n - 1) mod 14 + 1` rồi mô phỏng tối đa 14 ngày |
| 4 | Như Cách 2 nhưng trạng thái là số nguyên 8 bit và bước đi bằng thao tác bit |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Bảng băm trạng thái và nhảy qua chu kỳ (cách nộp chính)

**Giải thích từng đoạn.**

- `seen = {}`: ánh xạ `tuple(cells) -> số ngày đã trôi qua` khi gặp trạng thái đó.
- Vòng lặp `while n > 0`: nếu `key = tuple(cells)` đã có trong `seen` thì chu kỳ `p = seen[key] - n` (vì ta đếm ngược `n` còn lại: ngày hiện tại ứng với `n` còn lại nhỏ hơn); đặt `n %= p`; **chỉ nhảy một lần** (xóa `seen` hoặc dùng cờ) để tránh nhảy lại.
- Mỗi vòng: `seen[key] = n`; `cells = next_day(cells)`; `n -= 1`.

**Lưu ý quan trọng:** khóa phải là `tuple(cells)` (bản sao bất biến), không phải chính danh sách `cells` (sẽ bị thay đổi/không hashable).

In [6]:
def prison_cycle(cells: List[int], n: int) -> List[int]:
    cells = list(cells)
    seen = {}
    while n > 0:
        key = tuple(cells)
        if key in seen:
            period = seen[key] - n                 # số ngày giữa hai lần gặp trạng thái này
            n %= period
            seen = {}                              # đã nhảy chu kỳ: không nhảy lại
            if n == 0:
                break
        seen[key] = n
        cells = next_day(cells)
        n -= 1
    return cells

for cells, n, expected in examples:
    got = prison_cycle(cells, n)
    print(cells, n, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[0, 1, 0, 1, 1, 0, 0, 1] 7 -> [0, 0, 1, 1, 0, 0, 0, 0] OK
[1, 0, 0, 1, 0, 0, 1, 0] 1000000000 -> [0, 0, 1, 1, 1, 1, 1, 0] OK


**Code nộp LeetCode (Cách 2):**

In [7]:
class Solution:
    def prisonAfterNDays(self, cells: List[int], n: int) -> List[int]:
        seen = {}
        while n > 0:
            key = tuple(cells)
            if key in seen:
                n %= seen[key] - n
                seen = {}
                if n == 0:
                    break
            seen[key] = n
            nxt = [0] * 8
            for i in range(1, 7):
                nxt[i] = 1 if cells[i - 1] == cells[i + 1] else 0
            cells = nxt
            n -= 1
        return cells

sol_cycle = Solution().prisonAfterNDays
assert [sol_cycle(list(c), n) for c, n, _ in examples] == [e for _, _, e in examples]
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** thời gian `O(256 * 8)` (tối đa 256 trạng thái mỗi ngày tốn `O(8)`), bộ nhớ `O(256 * 8)`.

---

### 7.2. Cách 3 — Chu kỳ chia hết cho 14

**Ý tưởng khác.** Sau ngày `1`, dãy trạng thái có chu kỳ chia hết cho `14` (kiểm chứng ở mục 10). Vì vậy: mô phỏng **một ngày** (để ô đầu/cuối về `0`); với `n >= 1`, phần còn lại `n - 1` ngày có thể thay bằng `(n - 1) mod 14` ngày.

**Giải thích từng đoạn.** Công thức: `days = (n - 1) % 14 + 1` (số ngày thật sự cần mô phỏng, từ `1` đến `14`).

In [8]:
def prison_mod14(cells: List[int], n: int) -> List[int]:
    cells = list(cells)
    for _ in range((n - 1) % 14 + 1):
        cells = next_day(cells)
    return cells

for cells, n, expected in examples:
    got = prison_mod14(cells, n)
    print(cells, n, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[0, 1, 0, 1, 1, 0, 0, 1] 7 -> [0, 0, 1, 1, 0, 0, 0, 0] OK
[1, 0, 0, 1, 0, 0, 1, 0] 1000000000 -> [0, 0, 1, 1, 1, 1, 1, 0] OK


**Code nộp LeetCode (Cách 3):**

In [9]:
class Solution:
    def prisonAfterNDays(self, cells: List[int], n: int) -> List[int]:
        for _ in range((n - 1) % 14 + 1):
            cells = [0] + [1 if cells[i - 1] == cells[i + 1] else 0 for i in range(1, 7)] + [0]
        return cells

sol_mod14 = Solution().prisonAfterNDays
assert [sol_mod14(list(c), n) for c, n, _ in examples] == [e for _, _, e in examples]
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** `O(1)` (tối đa 14 ngày mô phỏng), `O(1)` bộ nhớ. Phụ thuộc vào tính chất chu kỳ chia hết cho `14`, nên cần kiểm chứng cẩn thận.

---

### 7.3. Cách 4 — Thao tác bit và phát hiện chu kỳ

**Ý tưởng khác.** Mã hóa trạng thái thành số nguyên 8 bit, bước đi `step_bits(s) = ~((s << 1) ^ (s >> 1)) & 0b01111110`, và phát hiện chu kỳ bằng danh sách `history` (trạng thái theo ngày) hoặc mảng `first_day[256]`. Khi gặp lại trạng thái ở ngày `b` (lần đầu ngày `a`), chu kỳ `p = b - a`; trạng thái ngày `n` là `history[a + (n - a) % p]`.

**Giải thích từng đoạn.** `first_day = [-1] * 256`; `history` lưu trạng thái ngày `0, 1, 2, ...`; dừng khi ngày hiện tại `d == n` hoặc gặp trạng thái cũ.

In [10]:
def prison_bits(cells: List[int], n: int) -> List[int]:
    s = to_int(cells)
    first_day = [-1] * 256
    history = []
    day = 0
    while True:
        if day == n:
            return to_cells(s)
        if first_day[s] != -1:
            a, p = first_day[s], day - first_day[s]
            return to_cells(history[a + (n - a) % p])
        first_day[s] = day
        history.append(s)
        s = step_bits(s)
        day += 1

for cells, n, expected in examples:
    got = prison_bits(cells, n)
    print(cells, n, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[0, 1, 0, 1, 1, 0, 0, 1] 7 -> [0, 0, 1, 1, 0, 0, 0, 0] OK
[1, 0, 0, 1, 0, 0, 1, 0] 1000000000 -> [0, 0, 1, 1, 1, 1, 1, 0] OK


**Code nộp LeetCode (Cách 4):**

In [11]:
class Solution:
    def prisonAfterNDays(self, cells: List[int], n: int) -> List[int]:
        s = sum(c << i for i, c in enumerate(cells))
        first_day = [-1] * 256
        history = []
        day = 0
        while True:
            if day == n:
                break
            if first_day[s] != -1:
                a, p = first_day[s], day - first_day[s]
                s = history[a + (n - a) % p]
                break
            first_day[s] = day
            history.append(s)
            s = ~((s << 1) ^ (s >> 1)) & 0b01111110
            day += 1
        return [(s >> i) & 1 for i in range(8)]

sol_bits = Solution().prisonAfterNDays
assert [sol_bits(list(c), n) for c, n, _ in examples] == [e for _, _, e in examples]
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** `O(256)` bước (mỗi bước `O(1)` bit), bộ nhớ `O(256)`.

## 8. TRACE từng bước

### 8.1. Cách 2 cho ví dụ 2: `[1,0,0,1,0,0,1,0]`, `n = 10^9` (phát hiện chu kỳ)

In [12]:
def trace_cycle(cells, n, limit=40):
    cells = list(cells)
    seen = {}
    day = 0
    while day < limit:
        key = tuple(cells)
        if key in seen:
            print(f"ngày {day}: gặp lại trạng thái {key} (lần đầu ngày {seen[key]}) -> chu kỳ {day - seen[key]}, vào chu kỳ ở ngày {seen[key]}")
            return seen[key], day - seen[key]
        seen[key] = day
        print(f"ngày {day:2}: {cells}")
        cells = next_day(cells)
        day += 1

a, p = trace_cycle(examples[1][0], 10 ** 9)
assert p == 14 or p in (1, 7, 14)

ngày  0: [1, 0, 0, 1, 0, 0, 1, 0]
ngày  1: [0, 0, 0, 1, 0, 0, 1, 0]
ngày  2: [0, 1, 0, 1, 0, 0, 1, 0]
ngày  3: [0, 1, 1, 1, 0, 0, 1, 0]
ngày  4: [0, 0, 1, 0, 0, 0, 1, 0]
ngày  5: [0, 0, 1, 0, 1, 0, 1, 0]
ngày  6: [0, 0, 1, 1, 1, 1, 1, 0]
ngày  7: [0, 0, 0, 1, 1, 1, 0, 0]
ngày  8: [0, 1, 0, 0, 1, 0, 0, 0]
ngày  9: [0, 1, 0, 0, 1, 0, 1, 0]
ngày 10: [0, 1, 0, 0, 1, 1, 1, 0]
ngày 11: [0, 1, 0, 0, 0, 1, 0, 0]
ngày 12: [0, 1, 0, 1, 0, 1, 0, 0]
ngày 13: [0, 1, 1, 1, 1, 1, 0, 0]
ngày 14: [0, 0, 1, 1, 1, 0, 0, 0]
ngày 15: gặp lại trạng thái (0, 0, 0, 1, 0, 0, 1, 0) (lần đầu ngày 1) -> chu kỳ 14, vào chu kỳ ở ngày 1


### 8.2. Nhảy qua chu kỳ cho ví dụ 2

Chu kỳ `p` vào ở ngày `a` (xem trên): trạng thái ngày `10^9` là trạng thái ở ngày `a + (10^9 - a) mod p`.

In [13]:
n = 10 ** 9
idx = a + (n - a) % p
print(f"a = {a}, p = {p}, ngày tương đương = {idx}")
assert prison_brute(examples[1][0], idx) == examples[1][2]

a = 1, p = 14, ngày tương đương = 6


### 8.3. Cách 3 cho ví dụ 2

`(10^9 - 1) % 14 + 1 = ?` rồi mô phỏng ngần ấy ngày.

In [14]:
days = (10 ** 9 - 1) % 14 + 1
print("số ngày mô phỏng:", days)
assert prison_brute(examples[1][0], days) == examples[1][2]

số ngày mô phỏng: 6


## 9. Code nộp LeetCode

Ta chọn **Cách 2** (bảng băm trạng thái và nhảy qua chu kỳ): tổng quát (không dựa vào hằng số `14`), `O(256 * 8)`.

```python
class Solution:
    def prisonAfterNDays(self, cells: List[int], n: int) -> List[int]:
        seen = {}
        while n > 0:
            key = tuple(cells)
            if key in seen:
                n %= seen[key] - n
                seen = {}
                if n == 0:
                    break
            seen[key] = n
            nxt = [0] * 8
            for i in range(1, 7):
                nxt[i] = 1 if cells[i - 1] == cells[i + 1] else 0
            cells = nxt
            n -= 1
        return cells
```

In [15]:
class Solution:
    def prisonAfterNDays(self, cells: List[int], n: int) -> List[int]:
        seen = {}
        while n > 0:
            key = tuple(cells)
            if key in seen:
                n %= seen[key] - n
                seen = {}
                if n == 0:
                    break
            seen[key] = n
            nxt = [0] * 8
            for i in range(1, 7):
                nxt[i] = 1 if cells[i - 1] == cells[i + 1] else 0
            cells = nxt
            n -= 1
        return cells

sol = Solution()
for cells, n, expected in examples:
    got = sol.prisonAfterNDays(list(cells), n)
    print(cells, n, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[0, 1, 0, 1, 1, 0, 0, 1] 7 -> [0, 0, 1, 1, 0, 0, 0, 0] OK
[1, 0, 0, 1, 0, 0, 1, 0] 1000000000 -> [0, 0, 1, 1, 1, 1, 1, 0] OK


## 10. Vì sao đúng

**(a) Hàm chuyển trạng thái.** Ngày kế tiếp chỉ phụ thuộc trạng thái hiện tại, nên có hàm `f` trên `256` trạng thái và `state_d = f^d(state_0)`.

**(b) Phát hiện chu kỳ (Cách 2, 4).** Dãy `state_0, state_1, ...` gồm các trạng thái thuộc tập hữu hạn nên có hai chỉ số `a < b` với `state_a = state_b` (nguyên lý chuồng bồ câu, `b <= 256`). Từ đó `state_{d + p} = state_d` với `p = b - a` và mọi `d >= a`. Vậy với `n >= a`: `state_n = state_{a + (n - a) mod p}`. Ở Cách 2, ta đếm ngược `n`: khi gặp lại trạng thái đã lưu với giá trị `n_cũ` thì số ngày giữa hai lần gặp là `n_cũ - n_hiện_tại = p`; còn lại `n` ngày và `n mod p` ngày đủ (vì sau `p` ngày trạng thái trở lại).

**(c) Nhận xét chu kỳ 14.** Sau ngày `1` mọi trạng thái có `cells[0] = cells[7] = 0`; chỉ còn `6` bit tự do. Ta kiểm chứng bằng máy: với **mọi 64** trạng thái có hai bit biên bằng `0`, `f^14(s) = s`? Điều này **không** đúng với mọi trạng thái (có trạng thái chu kỳ 1 hoặc 7 nhưng chu kỳ chia hết cho 14); ta kiểm tra `f^14(s) = s` với mọi `s` trong "ảnh" của `f` (các trạng thái sau ngày 1). Do đó `f^n(s) = f^(((n-1) mod 14) + 1)(s)` với `n >= 1`: viết `n = 1 + 14q + r`, `f^n(s) = f^(14q)(f^(1 + r)(s)) = f^(1 + r)(s)` vì `f^(1+r)(s)` thuộc ảnh của `f`.

**(d) Thao tác bit.** `step_bits` tính đúng `next_day` (kiểm tra trên mọi 256 trạng thái ở mục 3.3).

**Kiểm chứng bằng máy.** (1) Với **mọi 256** trạng thái ban đầu và mọi `n` từ `1` đến `60`, so mọi lời giải với mô phỏng từng ngày; (2) Oracle **độc lập** cho `n` rất lớn: **lũy thừa hàm bằng bình phương liên tiếp** (`f^n` = `f` hợp thành với chính nó `n` lần, tính bằng nhị phân của `n`, không mô phỏng từng ngày và không dùng chu kỳ); (3) kiểm tra tính chất chu kỳ chia hết cho `14` bằng cách tính chu kỳ thật của mọi trạng thái sau ngày `1`.

In [16]:
# hàm chuyển trạng thái dưới dạng bảng 256 phần tử và lũy thừa bằng bình phương liên tiếp
F = [step_bits(s) for s in range(256)]

def compose(g, h):                       # (g o h)(x) = g(h(x))
    return [g[h[x]] for x in range(256)]

def power_function(f, n):
    result = list(range(256))            # hàm đồng nhất
    base = f
    while n:
        if n & 1:
            result = compose(base, result)
        base = compose(base, base)
        n >>= 1
    return result

def oracle_power(cells: List[int], n: int) -> List[int]:
    return to_cells(power_function(F, n)[to_int(cells)])

assert oracle_power(*examples[0][:2]) == examples[0][2]
assert oracle_power(*examples[1][:2]) == examples[1][2]
print("oracle lũy thừa hàm khớp hai ví dụ (kể cả n = 10^9)")

# kiểm tra oracle lũy thừa với mô phỏng từng ngày trên mọi trạng thái, n <= 40
for s in range(256):
    cells = to_cells(s)
    cur = cells
    for n in range(1, 41):
        cur = next_day(cur)
        assert oracle_power(cells, n) == cur, (s, n)
print("oracle lũy thừa khớp mô phỏng từng ngày trên mọi 256 trạng thái và n = 1..40")

oracle lũy thừa hàm khớp hai ví dụ (kể cả n = 10^9)


oracle lũy thừa khớp mô phỏng từng ngày trên mọi 256 trạng thái và n = 1..40


In [17]:
# tính chu kỳ thật của mọi trạng thái sau ngày 1 (kiểm tra tính chất chu kỳ chia hết cho 14)
def true_period_and_prefix(s):
    seen = {}
    d = 0
    while s not in seen:
        seen[s] = d
        s = F[s]
        d += 1
    return seen[s], d - seen[s]

periods = set()
for s in range(256):
    a, p = true_period_and_prefix(F[s])               # trạng thái sau ngày 1
    periods.add(p)
    assert 14 % p == 0, (s, p)                        # chu kỳ chia hết cho... 14 chia hết cho p
print("các chu kỳ xuất hiện sau ngày 1:", sorted(periods), "(đều là ước của 14)")
for s in range(256):
    img = F[s]
    cur = img
    for _ in range(14):
        cur = F[cur]
    assert cur == img, s                              # f^14 là đồng nhất trên ảnh của f
print("f^14 là đồng nhất trên ảnh của f: kiểm tra xong với mọi trạng thái")

các chu kỳ xuất hiện sau ngày 1: [1, 7, 14] (đều là ước của 14)
f^14 là đồng nhất trên ảnh của f: kiểm tra xong với mọi trạng thái


Lưu ý: chu kỳ thật là một **ước** của `14` (`1`, `2`, `7` hoặc `14`); vì vậy "bước nhảy 14 ngày" luôn trả về cùng trạng thái, và công thức `((n - 1) mod 14) + 1` là đúng.

## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng từng ngày | `O(n)` | `O(1)` |
| 2. Bảng băm + nhảy chu kỳ | `O(256 * 8)` | `O(256 * 8)` |
| 3. `(n - 1) mod 14 + 1` | `O(14 * 8)` | `O(1)` |
| 4. Thao tác bit + chu kỳ | `O(256)` | `O(256)` |

Đo thử với `n = 10^9`:

In [18]:
rng = random.Random(11)
tests = [([rng.randint(0, 1) for _ in range(8)], 10 ** 9) for _ in range(2000)]
ref = [oracle_power(c, n) for c, n in tests]
for name, fn in (("Cách 2 bảng băm", sol_cycle), ("Cách 3 mod 14", sol_mod14), ("Cách 4 bit", sol_bits), ("nộp", Solution().prisonAfterNDays)):
    t0 = time.perf_counter()
    res = [fn(list(c), n) for c, n in tests]
    dt = (time.perf_counter() - t0) * 1000
    assert res == ref, name
    print(f"{name:16} 2000 truy vấn n = 10^9: {dt:8.1f} ms")

Cách 2 bảng băm  2000 truy vấn n = 10^9:     23.7 ms
Cách 3 mod 14    2000 truy vấn n = 10^9:      7.2 ms
Cách 4 bit       2000 truy vấn n = 10^9:      6.8 ms
nộp              2000 truy vấn n = 10^9:     23.4 ms


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai, đối chiếu mọi cách với oracle (mô phỏng từng ngày cho `n` nhỏ; lũy thừa hàm cho `n` lớn), duyệt **toàn bộ 256 trạng thái** và tính chất.

In [19]:
def solutions_list():
    return {
        "cycle": prison_cycle,
        "mod14": prison_mod14,
        "bits": prison_bits,
        "sol_cycle": sol_cycle,
        "sol_mod14": sol_mod14,
        "sol_bits": sol_bits,
        "final": Solution().prisonAfterNDays,
    }

def run_all(cells, n, expected=None, brute_limit=2000):
    ref = prison_brute(cells, n) if n <= brute_limit else oracle_power(cells, n)
    assert oracle_power(cells, n) == ref
    for name, fn in solutions_list().items():
        assert fn(list(cells), n) == ref, (name, cells, n)
    if expected is not None:
        assert ref == expected, ("kỳ vọng sai", cells, n, expected, ref)
    return ref

# 12.1 ví dụ đề
for cells, n, expected in examples:
    assert run_all(cells, n, expected) == expected
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([0, 0, 0, 0, 0, 0, 0, 0], 1, [0, 1, 1, 1, 1, 1, 1, 0]),        # mọi ô trong có hai ô kề bằng nhau (0,0)
    ([1, 1, 1, 1, 1, 1, 1, 1], 1, [0, 1, 1, 1, 1, 1, 1, 0]),        # hai ô kề (1,1): có người; ô biên về 0
    ([0, 1, 0, 1, 0, 1, 0, 1], 1, [0, 1, 1, 1, 1, 1, 1, 0]),        # xen kẽ: hai ô kề luôn bằng nhau
    ([1, 0, 0, 0, 0, 0, 0, 1], 1, [0, 0, 1, 1, 1, 1, 0, 0]),        # ô 1: (1,0) khác -> 0; ô 2..5: (0,0) -> 1; ô 6: (0,1) khác -> 0
    ([0, 1, 1, 0, 0, 0, 0, 0], 1, [0, 0, 0, 0, 1, 1, 1, 0]),        # ngày 2 của ví dụ 1
    ([0, 1, 0, 1, 1, 0, 0, 1], 1, [0, 1, 1, 0, 0, 0, 0, 0]),        # ngày 1 của ví dụ 1
    ([0, 1, 0, 1, 1, 0, 0, 1], 2, [0, 0, 0, 0, 1, 1, 1, 0]),        # ngày 2
    ([1, 0, 0, 1, 0, 0, 1, 0], 14, None),                            # kiểm bằng oracle (n = chu kỳ)
    ([0, 0, 1, 0, 0, 1, 0, 0], 10 ** 9, None),
    ([1, 1, 0, 0, 1, 1, 0, 0], 13, None),
]
for cells, n, expected in edge:
    run_all(cells, n, expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 10 ca


In [20]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm có seed rồi xác nhận lời giải đúng không bị lỗi)
def wrong_keep_ends(cells, n):                       # không đặt ô đầu/cuối về 0 (giữ nguyên giá trị cũ)
    cells = list(cells)
    for _ in range(n):
        new = cells[:]
        for i in range(1, 7):
            new[i] = 1 if cells[i - 1] == cells[i + 1] else 0
        cells = new
    return cells

def wrong_in_place(cells, n):                        # cập nhật tại chỗ (dùng giá trị đã đổi của ô bên trái)
    cells = list(cells)
    for _ in range(n):
        for i in range(1, 7):
            cells[i] = 1 if cells[i - 1] == cells[i + 1] else 0
        cells[0] = cells[7] = 0
    return cells

def wrong_plain_mod(cells, n):                       # n % 14 thô (n = 14 cho ngày 0)
    cells = list(cells)
    for _ in range(n % 14):
        cells = next_day(cells)
    return cells

def find_counter(wrong, tries=3000, seed=7):
    rng = random.Random(seed)
    for _ in range(tries):
        cells = [rng.randint(0, 1) for _ in range(8)]
        n = rng.randint(1, 30)
        if wrong(cells, n) != oracle_power(cells, n):
            return cells, n
    return None

counters = {}
for name, fn in (("không đặt ô biên về 0", wrong_keep_ends), ("cập nhật tại chỗ", wrong_in_place), ("n % 14 thô", wrong_plain_mod)):
    c = find_counter(fn)
    assert c is not None, name
    counters[name] = c
    cells, n = c
    print(f"{name:26} cells={cells} n={n}: sai -> {fn(cells, n)}, đúng -> {oracle_power(cells, n)}")
    run_all(cells, n)
print("12.3 phản ví dụ: OK")

không đặt ô biên về 0      cells=[1, 0, 1, 0, 0, 0, 1, 0] n=30: sai -> [1, 1, 0, 1, 1, 1, 1, 0], đúng -> [0, 0, 0, 1, 1, 1, 1, 0]
cập nhật tại chỗ           cells=[1, 0, 1, 0, 0, 0, 1, 0] n=30: sai -> [0, 1, 1, 0, 1, 1, 0, 0], đúng -> [0, 0, 0, 1, 1, 1, 1, 0]
n % 14 thô                 cells=[1, 0, 0, 1, 1, 1, 0, 1] n=28: sai -> [1, 0, 0, 1, 1, 1, 0, 1], đúng -> [0, 1, 1, 0, 0, 0, 1, 0]
12.3 phản ví dụ: OK


In [21]:
# 12.4 vét cạn: MỌI 256 trạng thái ban đầu x n = 1..60: mọi lời giải khớp mô phỏng từng ngày
t0 = time.perf_counter()
for s in range(256):
    cells = to_cells(s)
    cur = cells
    for n in range(1, 61):
        cur = next_day(cur)
        for name, fn in solutions_list().items():
            assert fn(list(cells), n) == cur, (name, cells, n)
print(f"12.4 mọi 256 trạng thái x n = 1..60 cho 7 lời giải: OK ({time.perf_counter() - t0:.1f} s)")

# 12.5 n lớn: mọi trạng thái x nhiều n lớn ngẫu nhiên và n quanh bội của 14, so với oracle lũy thừa hàm
rng = random.Random(5957)
for s in range(256):
    cells = to_cells(s)
    ns = [10 ** 9, 10 ** 9 - 1, 999999999, 14 * 10 ** 6, 14 * 10 ** 6 + 1, 14 * 10 ** 6 - 1] + [rng.randint(1, 10 ** 9) for _ in range(4)]
    for n in ns:
        ref = oracle_power(cells, n)
        for name, fn in solutions_list().items():
            assert fn(list(cells), n) == ref, (name, cells, n)
print("12.5 mọi 256 trạng thái x 10 giá trị n lớn: OK")

# 12.6 tính chất: sau mọi ngày n >= 1 ô đầu và ô cuối đều bằng 0
rng = random.Random(6957)
for _ in range(500):
    cells = [rng.randint(0, 1) for _ in range(8)]
    n = rng.randint(1, 10 ** 9)
    out = Solution().prisonAfterNDays(list(cells), n)
    assert out[0] == 0 and out[7] == 0
print("12.6 tính chất (ô biên = 0 sau mọi ngày n >= 1): OK")

# 12.7 đối xứng gương: đảo mảng rồi mô phỏng = mô phỏng rồi đảo
rng = random.Random(7957)
for _ in range(300):
    cells = [rng.randint(0, 1) for _ in range(8)]
    n = rng.randint(1, 10 ** 9)
    assert Solution().prisonAfterNDays(cells[::-1], n) == Solution().prisonAfterNDays(list(cells), n)[::-1]
print("12.7 đối xứng gương: OK")

12.4 mọi 256 trạng thái x n = 1..60 cho 7 lời giải: OK (1.1 s)


12.5 mọi 256 trạng thái x 10 giá trị n lớn: OK
12.6 tính chất (ô biên = 0 sau mọi ngày n >= 1): OK
12.7 đối xứng gương: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Không đặt ô đầu và ô cuối về `0`

Các ô biên không có hai ô kề nên **luôn trống** sau mỗi ngày. Giữ giá trị cũ cho kết quả sai.

In [22]:
cells, n = counters["không đặt ô biên về 0"]
print("cells =", cells, "n =", n, "| sai ->", wrong_keep_ends(cells, n), "| đúng ->", Solution().prisonAfterNDays(list(cells), n))
assert wrong_keep_ends(cells, n) != Solution().prisonAfterNDays(list(cells), n)

cells = [1, 0, 1, 0, 0, 0, 1, 0] n = 30 | sai -> [1, 1, 0, 1, 1, 1, 1, 0] | đúng -> [0, 0, 0, 1, 1, 1, 1, 0]


Vì sao sai: ví dụ 1 ngày 1 là `[0,1,1,0,0,0,0,0]` (ô `7` từ `1` về `0`). **Đúng:** `new[0] = new[7] = 0`.

### Lỗi 2 — Cập nhật tại chỗ

Tính ô `i` bằng ô `i - 1` **đã đổi** (của ngày mới) thay vì giá trị cũ.

In [23]:
cells, n = counters["cập nhật tại chỗ"]
print("cells =", cells, "n =", n, "| sai ->", wrong_in_place(cells, n), "| đúng ->", Solution().prisonAfterNDays(list(cells), n))
assert wrong_in_place(cells, n) != Solution().prisonAfterNDays(list(cells), n)

cells = [1, 0, 1, 0, 0, 0, 1, 0] n = 30 | sai -> [0, 1, 1, 0, 1, 1, 0, 0] | đúng -> [0, 0, 0, 1, 1, 1, 1, 0]


Vì sao sai: mọi ô của ngày mới phải tính từ trạng thái **ngày cũ**. **Đúng:** tạo danh sách mới `new` rồi gán.

### Lỗi 3 — `n % 14` thô

Trạng thái ngày `0` có thể **không thuộc chu kỳ** (ô biên có thể là `1`), nên `n % 14 = 0` không trả về ngày `14` mà trả về ngày `0`.

In [24]:
cells, n = counters["n % 14 thô"]
print("cells =", cells, "n =", n, "| sai ->", wrong_plain_mod(cells, n), "| đúng ->", Solution().prisonAfterNDays(list(cells), n))
assert wrong_plain_mod(cells, n) != Solution().prisonAfterNDays(list(cells), n)
print("n = 14 với ví dụ 2:", wrong_plain_mod(examples[1][0], 14), "(sai: giữ nguyên) so với", Solution().prisonAfterNDays(list(examples[1][0]), 14))

cells = [1, 0, 0, 1, 1, 1, 0, 1] n = 28 | sai -> [1, 0, 0, 1, 1, 1, 0, 1] | đúng -> [0, 1, 1, 0, 0, 0, 1, 0]
n = 14 với ví dụ 2: [1, 0, 0, 1, 0, 0, 1, 0] (sai: giữ nguyên) so với [0, 0, 1, 1, 1, 0, 0, 0]


Vì sao sai: chu kỳ chỉ bắt đầu từ ngày `1`. **Đúng:** `(n - 1) % 14 + 1`.

### Lỗi 4 — Mô phỏng `10^9` ngày

Dù mỗi ngày chỉ `8` ô, `10^9` ngày mất hàng chục giờ (xem mục 5). **Đúng:** phát hiện chu kỳ hoặc nhảy `mod 14`.

### Lỗi 5 — Lưu tham chiếu danh sách thay vì bản sao khi phát hiện chu kỳ

Dùng danh sách làm khóa từ điển gây `TypeError` (không băm được); lưu tham chiếu vào lịch sử rồi sửa tại chỗ làm mọi mục lịch sử đổi theo.

In [25]:
try:
    {[0, 1]: 0}
except TypeError as e:
    print("TypeError:", e)

state = [0, 1, 0, 1, 1, 0, 0, 1]
history = []
for _ in range(3):
    history.append(state)                 # lưu tham chiếu
    state[:] = next_day(state)            # sửa tại chỗ
print("lịch sử (cùng một đối tượng bị sửa):", history)
assert history[0] is history[1] is history[2]
print("đúng: khóa/bản sao bất biến tuple(cells) hoặc số nguyên 8 bit")

TypeError: unhashable type: 'list'
lịch sử (cùng một đối tượng bị sửa): [[0, 1, 1, 0, 0, 1, 0, 0], [0, 1, 1, 0, 0, 1, 0, 0], [0, 1, 1, 0, 0, 1, 0, 0]]
đúng: khóa/bản sao bất biến tuple(cells) hoặc số nguyên 8 bit


Vì sao sai: các mục lịch sử cùng trỏ tới một danh sách. **Đúng:** lưu `tuple(cells)` hoặc `to_int(cells)`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** mô phỏng với `n` khổng lồ trên **không gian trạng thái nhỏ** ⇒ **phát hiện chu kỳ**.
- **Khóa trạng thái:** `tuple(...)` hoặc số nguyên bit; dùng `seen[state] = ngày` rồi `n %= chu_kỳ`.
- **Tính hàm lặp nhiều lần:** lũy thừa hàm bằng bình phương liên tiếp (`f^n`).
- Ô biên có điều kiện đặc biệt ⇒ chú ý trạng thái ngày `0` có thể không thuộc chu kỳ.
- Cùng họ: Game of Life (289), Robot Bounded In Circle (1041), Happy Number (202), Linked List Cycle (141), Find the Duplicate Number (287).
- Test nên có: mọi `256` trạng thái, `n = 1`, `n` bội của `14`, `n = 10^9`.

## 15. Biến thể

1. **Số ô `k` bất kỳ** (không phải `8`): cùng quy tắc; chu kỳ có thể dài tới `2^(k-2)` (bài tập 5: tìm tiền chu kỳ và chu kỳ).
2. **Nhà tù vòng** (ô đầu kề ô cuối): không còn ô biên đặc biệt.
3. **Đếm số trạng thái khác nhau** trong một quỹ đạo.
4. Bài liên quan: Game of Life (289).

Dưới đây là biến thể 1 (độ dài `k` bất kỳ, phát hiện chu kỳ) kiểm bằng mô phỏng từng ngày.

In [26]:
def next_day_k(cells):
    k = len(cells)
    return [0] + [1 if cells[i - 1] == cells[i + 1] else 0 for i in range(1, k - 1)] + [0]

def prison_k(cells: List[int], n: int) -> List[int]:
    seen = {}
    cur = tuple(cells)
    history = []
    day = 0
    while day != n:
        if cur in seen:
            a, p = seen[cur], day - seen[cur]
            return list(history[a + (n - a) % p])
        seen[cur] = day
        history.append(cur)
        cur = tuple(next_day_k(list(cur)))
        day += 1
    return list(cur)

rng = random.Random(15)
for _ in range(300):
    k = rng.randint(3, 10)
    cells = [rng.randint(0, 1) for _ in range(k)]
    n = rng.randint(1, 3000)
    ref = cells
    for _ in range(n):
        ref = next_day_k(ref)
    assert prison_k(cells, n) == ref, (cells, n)
assert prison_k(examples[1][0], 10 ** 9) == examples[1][2]
print("biến thể độ dài k bất kỳ khớp mô phỏng; k = 8 khớp ví dụ 2")

biến thể độ dài k bất kỳ khớp mô phỏng; k = 8 khớp ví dụ 2


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng | Từng ngày | `O(n)` | `O(1)` | Oracle, `n` nhỏ |
| 2. Bảng băm + chu kỳ | `seen[state]` | `O(256 * 8)` | `O(256 * 8)` | **Cách nộp chính**, tổng quát |
| 3. `(n - 1) mod 14 + 1` | Chu kỳ chia hết 14 | `O(1)` | `O(1)` | Ngắn nhất, cần chứng minh |
| 4. Bit + chu kỳ | 8 bit | `O(256)` | `O(256)` | Nhanh, gọn |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng, rồi chạy để kiểm tra:

```python
print(next_day([1, 1, 1, 1, 1, 1, 1, 1]))
print(next_day([0, 0, 0, 0, 0, 0, 0, 0]))
print((10 ** 9 - 1) % 14 + 1)
```

In [27]:
# Bài 1: ghi dự đoán của bạn
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `cells = [1, 0, 0, 1, 0, 0, 1, 0]`: ghi trạng thái sau ngày `1` và ngày `2`.

In [28]:
# Bài 2: điền kết quả tính tay
hand_day1 = None
hand_day2 = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành phát hiện chu kỳ bằng bảng băm.

In [29]:
def exercise_3_cycle(cells: List[int], n: int) -> List[int]:
    seen = {}
    while n > 0:
        key = tuple(cells)
        # TODO 1: nếu key đã có trong seen thì n %= (seen[key] - n) và đặt lại seen = {}
        seen[key] = n
        # TODO 2: tính trạng thái ngày kế tiếp (ô đầu và ô cuối bằng 0), giảm n
        n -= 1
    return cells


# for cells, n, expected in examples:
#     assert exercise_3_cycle(list(cells), n) == expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, hãy tự viết `exercise_4_bits(cells, n)` dùng số nguyên 8 bit và `step = ~((s << 1) ^ (s >> 1)) & 0b01111110`.

In [30]:
def exercise_4_bits(cells: List[int], n: int) -> List[int]:
    # TODO
    pass


# for cells, n, expected in examples:
#     assert exercise_4_bits(list(cells), n) == expected

### Bài 5 — Mini challenge: tiền chu kỳ và chu kỳ (khó)

Viết `exercise_5_cycle_info(cells)` trả về cặp `(a, p)`: `a` là ngày đầu tiên mà trạng thái bắt đầu thuộc chu kỳ và `p` là độ dài chu kỳ (với quy tắc nhà tù 8 ô). Kiểm tra bằng mô phỏng.

```python
exercise_5_cycle_info([0, 1, 0, 1, 1, 0, 0, 1])  -> (1, 14)   # ngày 0 không thuộc chu kỳ (ô biên = 1)
exercise_5_cycle_info([0, 0, 0, 0, 0, 0, 0, 0])  -> (0, 7)    # trạng thái toàn 0 thuộc chu kỳ độ dài 7
```
Gợi ý: dùng `first_day` như Cách 4; khi gặp lại trạng thái ở ngày `d` mà lần đầu ở ngày `a` thì `p = d - a`.

In [31]:
def exercise_5_cycle_info(cells: List[int]):
    # TODO
    pass

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: mọi ô trong có hai ô kề giống nhau nên thành `1`; ô biên về `0`.
- Mức 2: `[1]*8` và `[0]*8` đều cho `[0, 1, 1, 1, 1, 1, 1, 0]`; `10^9 - 1 = 999999999`, chia `14`...
- Mức 3: `[0,1,1,1,1,1,1,0]`, `[0,1,1,1,1,1,1,0]`, `6`.

**Bài 2**
- Mức 1: ô `i` là `1` nếu `cells[i - 1] == cells[i + 1]`.
- Mức 2: ngày 1 từ `[1,0,0,1,0,0,1,0]`: ô1: `(1,0)` khác -> 0; ô2: `(0,1)` -> 0; ô3: `(0,0)` -> 1; ô4: `(1,0)` -> 0; ô5: `(0,1)` -> 0; ô6: `(0,0)` -> 1.
- Mức 3: ngày 1 `[0,0,0,1,0,0,1,0]`; ngày 2 `[0,1,0,1,0,0,1,0]`.

**Bài 3**
- Mức 1: `if key in seen: n %= seen[key] - n; seen = {}`.
- Mức 2: nếu `n == 0` sau khi lấy dư thì thoát.
- Mức 3: `cells = [0] + [...] + [0]`; `n -= 1`.

**Bài 4**
- Mức 1: `s = sum(c << i ...)`.
- Mức 2: `step = ~((s << 1) ^ (s >> 1)) & 0b01111110`.
- Mức 3: dùng `first_day[256]` và `history`; `history[a + (n - a) % p]`.

**Bài 5**
- Mức 1: mô phỏng ngày `0, 1, 2, ...` lưu `first_day[state]`.
- Mức 2: khi gặp lại ở ngày `d`: `a = first_day[state]`, `p = d - a`.
- Mức 3: trả `(a, p)`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [32]:
print(next_day([1, 1, 1, 1, 1, 1, 1, 1]))
print(next_day([0, 0, 0, 0, 0, 0, 0, 0]))
print((10 ** 9 - 1) % 14 + 1)

assert next_day([1] * 8) == [0, 1, 1, 1, 1, 1, 1, 0]
assert next_day([0] * 8) == [0, 1, 1, 1, 1, 1, 1, 0]
assert (10 ** 9 - 1) % 14 + 1 == 6

[0, 1, 1, 1, 1, 1, 1, 0]
[0, 1, 1, 1, 1, 1, 1, 0]
6


### Đáp án Bài 2

Ngày 1: `[0,0,0,1,0,0,1,0]` (ô 3 và ô 6 có hai ô kề cùng `0`); ngày 2: `[0,1,0,1,0,0,1,0]`.

In [33]:
cells = [1, 0, 0, 1, 0, 0, 1, 0]
hand_day1_solution = [0, 0, 0, 1, 0, 0, 1, 0]
hand_day2_solution = [0, 1, 0, 1, 0, 0, 1, 0]
print("ngày 1:", hand_day1_solution, "| ngày 2:", hand_day2_solution)
assert hand_day1_solution == oracle_power(cells, 1) == next_day(cells)
assert hand_day2_solution == oracle_power(cells, 2)
print("Bài 2 OK")

ngày 1: [0, 0, 0, 1, 0, 0, 1, 0] | ngày 2: [0, 1, 0, 1, 0, 0, 1, 0]
Bài 2 OK


### Đáp án Bài 3

In [34]:
def exercise_3_cycle_solution(cells: List[int], n: int) -> List[int]:
    seen = {}
    while n > 0:
        key = tuple(cells)
        if key in seen:
            n %= seen[key] - n
            seen = {}
            if n == 0:
                break
        seen[key] = n
        cells = [0] + [1 if cells[i - 1] == cells[i + 1] else 0 for i in range(1, 7)] + [0]
        n -= 1
    return cells

for cells, n, expected in examples:
    assert exercise_3_cycle_solution(list(cells), n) == expected
rng = random.Random(3)
for _ in range(300):
    cells = [rng.randint(0, 1) for _ in range(8)]
    n = rng.randint(1, 10 ** 9)
    assert exercise_3_cycle_solution(list(cells), n) == oracle_power(cells, n)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [35]:
def exercise_4_bits_solution(cells: List[int], n: int) -> List[int]:
    s = sum(c << i for i, c in enumerate(cells))
    first_day = [-1] * 256
    history = []
    day = 0
    while day != n:
        if first_day[s] != -1:
            a, p = first_day[s], day - first_day[s]
            s = history[a + (n - a) % p]
            break
        first_day[s] = day
        history.append(s)
        s = ~((s << 1) ^ (s >> 1)) & 0b01111110
        day += 1
    return [(s >> i) & 1 for i in range(8)]

for cells, n, expected in examples:
    assert exercise_4_bits_solution(list(cells), n) == expected
rng = random.Random(4)
for _ in range(300):
    cells = [rng.randint(0, 1) for _ in range(8)]
    n = rng.randint(1, 10 ** 9)
    assert exercise_4_bits_solution(list(cells), n) == oracle_power(cells, n)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [36]:
def exercise_5_cycle_info_solution(cells: List[int]):
    s = to_int(cells)
    first_day = {}
    day = 0
    while s not in first_day:
        first_day[s] = day
        s = step_bits(s)
        day += 1
    return first_day[s], day - first_day[s]

def cycle_info_oracle(cells, limit=600):
    """Tìm (a, p) bằng mô phỏng thuần: tìm a, p nhỏ nhất sao cho trạng thái ngày a + p bằng ngày a (so sánh trong 600 ngày)."""
    states = [list(cells)]
    for _ in range(limit):
        states.append(next_day(states[-1]))
    for a in range(0, 300):
        for p in range(1, 300):
            if all(states[a + t] == states[a + t + p] for t in range(0, 30)) and states[a] == states[a + p]:
                return a, p
    return None

assert exercise_5_cycle_info_solution([0, 1, 0, 1, 1, 0, 0, 1]) == (1, 14)
assert exercise_5_cycle_info_solution([0] * 8) == (0, 7)
print("cells ví dụ 1:", exercise_5_cycle_info_solution([0, 1, 0, 1, 1, 0, 0, 1]), "| toàn 0:", exercise_5_cycle_info_solution([0] * 8))
rng = random.Random(5)
for _ in range(100):
    cells = [rng.randint(0, 1) for _ in range(8)]
    assert exercise_5_cycle_info_solution(cells) == cycle_info_oracle(cells), cells
print("Bài 5 OK")

cells ví dụ 1: (1, 14) | toàn 0: (0, 7)
Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (phát hiện chu kỳ bằng bảng băm)

```python
class Solution:
    def prisonAfterNDays(self, cells: List[int], n: int) -> List[int]:
        seen = {}                                          # tuple(cells) -> n còn lại lúc gặp
        while n > 0:
            key = tuple(cells)
            if key in seen:                                # gặp lại: chu kỳ = seen[key] - n
                n %= seen[key] - n
                seen = {}                                  # chỉ nhảy một lần
                if n == 0:
                    break
            seen[key] = n
            nxt = [0] * 8                                  # ô biên luôn 0; tính từ trạng thái CŨ
            for i in range(1, 7):
                nxt[i] = 1 if cells[i - 1] == cells[i + 1] else 0
            cells = nxt
            n -= 1
        return cells
```

Công thức nhớ nhanh: tối đa `256` trạng thái ⇒ có chu kỳ; sau ngày 1 chu kỳ chia hết cho 14: `n = (n - 1) % 14 + 1`.

### Checklist trước khi submit

- [ ] Ô đầu và ô cuối đặt về `0` sau mỗi ngày?
- [ ] Tính ngày mới từ trạng thái **cũ** (không cập nhật tại chỗ)?
- [ ] Khóa chu kỳ là `tuple(cells)` (bản sao bất biến), không phải danh sách bị sửa tại chỗ?
- [ ] Không dùng `n % 14` thô (dùng `(n - 1) % 14 + 1`)?
- [ ] Không mô phỏng `10^9` ngày?
- [ ] Đã thử: mọi trạng thái, `n = 1`, bội của `14`, `n = 10^9`?
- [ ] Tên method đúng `prisonAfterNDays(self, cells, n)` và trả `List[int]`?